## Proyecto Final: Investigación profunda del dataset

#### Importar paquete y cargar ds
-Se aplica keep_default_na=False, para conservar los textos literales, sin convertirlos en nulos.

In [308]:
import pandas as pd
import numpy as np 
from limpieza import limpiar

df = pd.read_csv("data/pagos_fintech.csv", keep_default_na=False)

productos = pd.read_csv("data/productos.csv", keep_default_na=False)

regiones = pd.read_csv("data/regiones.csv", keep_default_na=False)


#### Datos básicos del dataset
- Dimensiones del dataset principal y relacionados

In [309]:
print("Dimensiones de pagos:", df.shape)
print("Dimensiones de productos:", productos.shape)
print("Dimensiones de regiones:", regiones.shape)

display(df.head())
display(productos)
display(regiones)

Dimensiones de pagos: (100000, 14)
Dimensiones de productos: (6, 4)
Dimensiones de regiones: (4, 3)


,payment_id,customer_id,product_id,payment_amount,payment_status,currency,pay_method,pay_dt,region_id,status_customer,risk_segment,kyc_level,income_bracket,signup_dt
0,5246022,5246023,1,31.99,paid,USD,card,2024-02-15 10:07:02,4,Active,Low,None,High,2023-09-01
1,6742278,6742279,1,47.99,pending,USD,card,2024-02-09 14:11:18,4,Suspended,High,Full,High,2024-12-30
2,6087809,6087810,6,18.99,paidd,USD,cash,2024-02-02 00:23:29,3,Active,Low,None,Low,2023-12-07
3,7484560,7484561,5,9.99,paid,USD,transfer,2024-02-18 04:22:40,2,Active,Low,None,Medium,2024-08-24
4,10072317,10072318,4,46.99,paid,USD,card,2024-02-01 20:05:17,3,Suspended,High,Full,High,2024-05-23


,product_id,product_type,interest_rate,currency
0,1,Wallet,0.00,USD
1,2,Card,0.24,USD
2,3,Loan,0.35,USD
3,4,Loan,0.18,USD
4,5,Card,0.29,USD
5,6,Wallet,0.00,USD


,region_id,region_name,country
0,1,Caribe,DO
1,2,Andina,CO
2,3,Europa,ES
3,4,Norteamérica,US


In [310]:
df["kyc_level"].value_counts(dropna=False)

kyc_level
None     41061
Basic    35087
Full     17730
          6122
Name: count, dtype: int64

#### Diagnóstico 

In [311]:
def diagnosticar(df):
    """
    Resume los tipos de datos, los valores nulos,
    el porcentaje de nulos y los valores únicos.
    """
    nulos = df.isna().sum() 
    porcentaje_nulos = (nulos / len(df)) * 100

    diagnostico = pd.DataFrame({
        "tipo_dato": df.dtypes.astype(str),
        "nulos": nulos,
        "porcentaje_nulos": porcentaje_nulos.round(2),
        "valores_unicos": df.nunique()
    })

    # Payment_id repetidos.
    ids = df.loc[df["payment_id"].notna(), "payment_id"]
    repetidos = ids.duplicated().sum()

    print("\nCantidad de payment_id repetidos:", repetidos)

    return diagnostico


display(diagnosticar(df))


Cantidad de payment_id repetidos: 250


,tipo_dato,nulos,porcentaje_nulos,valores_unicos
payment_id,int64,0,0.0,99750
customer_id,int64,0,0.0,99960
product_id,int64,0,0.0,6
payment_amount,str,0,0.0,43
payment_status,str,0,0.0,5
currency,str,0,0.0,3
pay_method,str,0,0.0,3
pay_dt,str,0,0.0,91859
region_id,str,0,0.0,5
status_customer,str,0,0.0,4


##### Comentario:
- Se observa que payment_amount se registra como str, por lo que requiere validación.
- pay_dt y signup_dt se registran como str, no como variables con tipo de dato de datetime, se requiere validación.

#### Investigación de payment_amount

In [312]:
print("Tipo de dato original:", df["payment_amount"].dtype)

# Se convierte str directamente a números
monto = pd.to_numeric(df["payment_amount"], errors="coerce")

print("Nulos originales:", df["payment_amount"].isna().sum())
print("Valores no convertibles:", monto.isna().sum())

# Valores que no pudieron convertirse
valores_no_convertibles = df.loc[monto.isna() & df["payment_amount"].ne(""), "payment_amount"]

print("\nValores que no se pudieron convertir:")
display(valores_no_convertibles.value_counts().to_frame("cantidad"))


Tipo de dato original: str
Nulos originales: 0
Valores no convertibles: 6019

Valores que no se pudieron convertir:


,cantidad
payment_amount,
"12,50",6019


##### Comentario
- Hay 6,019 valores con el texto 12,50. Se puede suponer que la coma (,) forza el tipo de dato a leerse como str.

In [313]:
# Reemplazo de coma decimal por punto

monto_corregido = pd.to_numeric(df["payment_amount"].astype("string").str.replace(",", ".", regex=False), errors="coerce")

print("Valores no convertibles antes:", monto.isna().sum())
print("Valores no convertibles después:", monto_corregido.isna().sum())

# Comprobar la conversión de 12,50
print("\nEjemplo de conversión:")
display(
    pd.DataFrame({
        "original": df["payment_amount"].loc[df["payment_amount"].eq("12,50")].head(),
        "convertido": monto_corregido.loc[df["payment_amount"].eq("12,50")].head()
    })
)

Valores no convertibles antes: 6019
Valores no convertibles después: 0

Ejemplo de conversión:


,original,convertido
7,"12,50",12.5
11,"12,50",12.5
18,"12,50",12.5
29,"12,50",12.5
37,"12,50",12.5


In [314]:
# Tipo de dato original
print("Tipo original de payment_amount:", df["payment_amount"].dtype)

# Tipo de dato después de la conversión temporal
print("Tipo de dato de monto_corregido:", monto_corregido.dtype)

Tipo original de payment_amount: str
Tipo de dato de monto_corregido: Float64


##### Comentario
- Los 6,019 valores que no se podían convertir corresponden al texto 12,50. Al sustituir la coma por un punto, 12,50 se convierte en 12.5 y desaparecen los errores de conversión.

#### Investigación de payment_status

In [315]:
print("Estados de pago y sus frecuencias:")
display(df["payment_status"].value_counts(dropna=False).to_frame("cantidad"))

Estados de pago y sus frecuencias:


,cantidad
payment_status,
paid,73427
failed,9792
pending,8304
paidd,6030
chargeback,2447


##### Comentario
- Existe el registro de "paidd" que sugiere un typo. Requiere validación y correlación con las demás variables/tablas para descartar patrones de dispersión. 

In [316]:
# payment_status por método de pago
tabla_estado_metodo = pd.crosstab(df["payment_status"], df["pay_method"], margins=True)

display(tabla_estado_metodo)

pay_method,card,cash,transfer,All
payment_status,,,,
chargeback,946,728,773,2447
failed,3780,2994,3018,9792
paid,31217,21008,21202,73427
paidd,1994,2024,2012,6030
pending,3192,2578,2534,8304
All,41129,29332,29539,100000


In [317]:
# payment_status por moneda
tabla_estado_moneda = pd.crosstab(df["payment_status"], df["currency"], margins=True)

display(tabla_estado_moneda)

currency,EURO,USD,USDT,All
payment_status,,,,
chargeback,146,2266,35,2447
failed,566,9070,156,9792
paid,3615,68764,1048,73427
paidd,19,6003,8,6030
pending,459,7703,142,8304
All,4805,93806,1389,100000


In [318]:
# payment_status vs tipo de producto
pagos_productos = df.merge(productos[["product_id", "product_type"]], on="product_id", how="left", validate="many_to_one")

tabla_estado_producto = pd.crosstab(pagos_productos["payment_status"], pagos_productos["product_type"], margins=True)

display(tabla_estado_producto)

product_type,Card,Loan,Wallet,All
payment_status,,,,
chargeback,773,865,809,2447
failed,3018,3376,3398,9792
paid,21202,23439,28786,73427
paidd,2012,2001,2017,6030
pending,2534,2901,2869,8304
All,29539,32582,37879,100000


In [319]:
# Distribución porcentual por método de pago
tabla_estado_metodo_pct = pd.crosstab(df["payment_status"], df["pay_method"], normalize="columns").mul(100).round(2)

display(tabla_estado_metodo_pct)

pay_method,card,cash,transfer
payment_status,,,
chargeback,2.30,2.48,2.62
failed,9.19,10.21,10.22
paid,75.90,71.62,71.78
paidd,4.85,6.90,6.81
pending,7.76,8.79,8.58


##### Comentario
- Los registros paidd aparecen en los tres métodos de pago. No se concentran exclusivamente en uno.
- paidd aparece en las tres monedas. 6,003 de los 6,030 registros están en USD.
- paidd aparece en los tres tipos de producto, con frecuencias similares.

In [320]:

# Imputación temporal
payment_status_temporal = df["payment_status"].replace({"paidd": "paid"})

# Antes y después
comparacion_estados = pd.DataFrame({
    "antes": df["payment_status"].value_counts(dropna=False),
    "despues": payment_status_temporal.value_counts(dropna=False)
}).fillna(0).astype(int)

comparacion_estados["diferencia"] = (comparacion_estados["despues"] - comparacion_estados["antes"])

display(comparacion_estados)

# Registros que cambiarían
registros_modificados = (df["payment_status"] != payment_status_temporal).sum()

print("\nRegistros que cambiarían:", registros_modificados)
print("Filas originales:", len(df))
print("Filas después de la simulación:", len(payment_status_temporal))


,antes,despues,diferencia
payment_status,,,
chargeback,2447,2447,0
failed,9792,9792,0
paid,73427,79457,6030
paidd,6030,0,-6030
pending,8304,8304,0



Registros que cambiarían: 6030
Filas originales: 100000
Filas después de la simulación: 100000


##### Comentario
- Al realizar la imputación prueba, se unifican dos categorías en paid y no cambia la cantidad de transacciones.

#### Investigación de pagos sin clientes

In [321]:
cadenas_vacias = df.eq("").sum()

columnas_con_vacios = (
    cadenas_vacias[cadenas_vacias > 0]
    .sort_values(ascending=False)
)

print("Columnas con cadenas vacías y cantidad de registros:")
display(columnas_con_vacios.to_frame("cantidad"))

Columnas con cadenas vacías y cantidad de registros:


,cantidad
region_id,6122
status_customer,6122
risk_segment,6122
kyc_level,6122
income_bracket,6122
signup_dt,6122


In [322]:
campos_perfil = columnas_con_vacios.index.tolist()

print("Columnas identificadas:", campos_perfil)

# Comprobar las filas con todos esos campos vacíos
perfil_vacio = df[campos_perfil].eq("")
sin_perfil = perfil_vacio.all(axis=1)

print("\nPagos con todos los campos vacíos:", sin_perfil.sum())
print("Porcentaje del dataset:", round(sin_perfil.mean() * 100, 2), "%")

display(df.loc[sin_perfil].head(10))

Columnas identificadas: ['region_id', 'status_customer', 'risk_segment', 'kyc_level', 'income_bracket', 'signup_dt']

Pagos con todos los campos vacíos: 6122
Porcentaje del dataset: 6.12 %


,payment_id,customer_id,product_id,payment_amount,payment_status,currency,pay_method,pay_dt,region_id,status_customer,risk_segment,kyc_level,income_bracket,signup_dt
38,5971700,105971700,3,29.99,paid,USD,cash,2024-02-23 19:41:40,,,,,,
51,9382100,109382100,3,29.99,paid,USD,cash,2024-02-16 23:55:00,,,,,,
73,7553700,107553700,1,-25.00,paid,USD,card,2024-02-18 23:35:00,,,,,,
93,8605800,108605800,1,-25.00,paid,USD,card,2024-02-08 00:16:40,,,,,,
124,6349700,106349700,3,29.99,paid,USD,cash,2024-02-05 01:08:20,,,,,,
126,7616700,107616700,1,-25.00,paid,USD,card,2024-02-19 17:05:00,,,,,,
132,5765900,105765900,3,29.99,paid,USD,cash,2024-02-21 10:31:40,,,,,,
157,5191900,105191900,5,29.99,paid,USD,transfer,2024-02-14 19:05:00,,,,,,
197,5596500,105596500,1,29.99,paid,USD,card,2024-02-19 11:28:20,,,,,,
209,6004600,106004600,5,9.99,paid,USD,transfer,2024-02-01 01:16:40,,,,,,


##### Comentario
- 6 columnas tienen cadenas vacías: region_id, status_customer, risk_segment, kyc_level, income_bracket y signup_dt.
- Cada columna tiene exactamente 6,122 valores vacíos.
- Al comprobar las filas, encontramos que los seis campos están vacíos simultáneamente en 6,122 pagos, equivalentes al 6.12% del dataset.

In [323]:
# Estados de los pagos sin información del perfil
estados_sin_perfil = (df.loc[sin_perfil, "payment_status"].value_counts(dropna=False))

display(estados_sin_perfil.to_frame("cantidad"))

,cantidad
payment_status,
paid,6109
paidd,13


In [324]:
# Se compara la tasa de fallidos antes y después de excluir
# temporalmente los pagos sin información del perfil

fallido = df["payment_status"].isin(["failed", "chargeback"])

comparacion_perfil = pd.DataFrame({
    "grupo": [
        "Todos los pagos",
        "Excluyendo pagos sin perfil"
    ],
    "total_pagos": [
        len(df),
        int((~sin_perfil).sum())
    ],
    "pagos_fallidos": [
        int(fallido.sum()),
        int(fallido.loc[~sin_perfil].sum())
    ],
    "tasa_fallidos_pct": [
        round(fallido.mean() * 100, 2),
        round(fallido.loc[~sin_perfil].mean() * 100, 2)
    ]
})

display(comparacion_perfil)

,grupo,total_pagos,pagos_fallidos,tasa_fallidos_pct
0,Todos los pagos,100000,12239,12.24
1,Excluyendo pagos sin perfil,93878,12239,13.04


##### Comentario
- 6,122 pagos (6.12% del dataset) en los que los seis campos de información del perfil están vacíos simultáneamente. Sin embargo, los registros conservan el customer_id y datos de la transacción. De estos pagos, 6,109 tienen estado paid y 13 tienen estado paidd.
- Al excluirlos temporalmente, la tasa de fallidos aumenta del 12.24% al 13.04%, una diferencia de 0.80 puntos porcentuales, mientras el número de pagos fallidos permanece en 12,239.

In [325]:

# Imputación temporal con relleno de campos vacíos con "Desconocido"
df_perfil_temporal = df.copy()

campos_perfil = [
    "region_id",
    "status_customer",
    "risk_segment",
    "kyc_level",
    "income_bracket",
    "signup_dt"
]

# Rellenar únicamente las cadenas vacías en los campos del perfil
df_perfil_temporal[campos_perfil] = (df_perfil_temporal[campos_perfil].replace("", "Desconocido"))

# Resultado del relleno
print("Cadenas vacías antes:")
display(df[campos_perfil].eq("").sum().to_frame("cantidad"))

print("\nCadenas vacías después:")
display(df_perfil_temporal[campos_perfil].eq("").sum().to_frame("cantidad"))

print("\nValores 'Desconocido' después del relleno:")
display(df_perfil_temporal[campos_perfil].eq("Desconocido").sum().to_frame("cantidad"))

# Cantidad de filas antes y después
print("\nFilas originales:", len(df))
print("Filas después de la simulación:", len(df_perfil_temporal))


Cadenas vacías antes:


,cantidad
region_id,6122
status_customer,6122
risk_segment,6122
kyc_level,6122
income_bracket,6122
signup_dt,6122



Cadenas vacías después:


,cantidad
region_id,0
status_customer,0
risk_segment,0
kyc_level,0
income_bracket,0
signup_dt,0



Valores 'Desconocido' después del relleno:


,cantidad
region_id,6122
status_customer,6122
risk_segment,6122
kyc_level,6122
income_bracket,6122
signup_dt,6122



Filas originales: 100000
Filas después de la simulación: 100000


In [326]:
# Tasa de fallidos antes y después del relleno
fallido_original = df["payment_status"].isin(["failed", "chargeback"])

fallido_relleno = df_perfil_temporal["payment_status"].isin(["failed", "chargeback"])

comparacion_relleno = pd.DataFrame({
    "escenario": [
        "Dataset original",
        "Perfil rellenado con Desconocido"
    ],
    "total_pagos": [
        len(df),
        len(df_perfil_temporal)
    ],
    "pagos_fallidos": [
        int(fallido_original.sum()),
        int(fallido_relleno.sum())
    ],
    "tasa_fallidos_pct": [
        round(fallido_original.mean() * 100, 2),
        round(fallido_relleno.mean() * 100, 2)
    ]
})

display(comparacion_relleno)

,escenario,total_pagos,pagos_fallidos,tasa_fallidos_pct
0,Dataset original,100000,12239,12.24
1,Perfil rellenado con Desconocido,100000,12239,12.24


##### Comentario
-Al simular el reemplazo de las cadenas vacías por «Desconocido» en los seis campos del perfil. Las 6,122 cadenas vacías por columna fueron reemplazadas, sin modificar el total de 100,000 filas.
La tasa de fallidos permaneció en 12.24% y el número de pagos fallidos se mantuvo en 12,239. Esto confirma que el relleno no afecta directamente este indicador, ya que no se modificaron los estados de los pagos.

#### Investigación de montos negativos 

In [327]:
monto_analisis = pd.to_numeric(df["payment_amount"].astype("string").str.replace(",", ".", regex=False), errors="coerce")

# Identificar los negativos
negativos = monto_analisis < 0
pagos_negativos = df.loc[negativos].copy()

print("Cantidad de pagos con importes negativos:", len(pagos_negativos))
print("Porcentaje del dataset:", round(len(pagos_negativos) / len(df) * 100, 2), "%")

print("\nDistribución de los importes negativos:")
display(monto_analisis.loc[negativos].value_counts().sort_index().to_frame("cantidad"))

Cantidad de pagos con importes negativos: 6328
Porcentaje del dataset: 6.33 %

Distribución de los importes negativos:


,cantidad
payment_amount,
-25.0,6328


In [328]:

# Negativos vs variables de transacción

columnas_revision = [
    "payment_status",
    "product_id",
    "pay_method",
    "currency"
]

for columna in columnas_revision:
    if columna in pagos_negativos.columns:
        print(f"\nDistribución de {columna}:")
        display(
            pagos_negativos[columna]
            .value_counts(dropna=False)
            .to_frame("cantidad")
        )


Distribución de payment_status:


,cantidad
payment_status,
paid,6311
paidd,17



Distribución de product_id:


,cantidad
product_id,
1,6328



Distribución de pay_method:


,cantidad
pay_method,
card,6328



Distribución de currency:


,cantidad
currency,
USD,6122
EURO,206


In [329]:

# Cantidad y suma de negativos vs no negativos
comparacion_importes = pd.DataFrame({
    "grupo": ["Importes negativos", "Importes cero o positivos"],
    "cantidad_pagos": [
        int(negativos.sum()),
        int((~negativos & monto_analisis.notna()).sum())
    ],
    "suma_importes": [
        monto_analisis.loc[negativos].sum(),
        monto_analisis.loc[
            (~negativos) & monto_analisis.notna()
        ].sum()
    ],
    "importe_mediano": [
        monto_analisis.loc[negativos].median(),
        monto_analisis.loc[
            (~negativos) & monto_analisis.notna()
        ].median()
    ]
})

display(comparacion_importes)

,grupo,cantidad_pagos,suma_importes,importe_mediano
0,Importes negativos,6328,-158200.00,-25.00
1,Importes cero o positivos,93672,16447643.74,27.99


##### Comentario
- Todos los negativos son de −25.00, corresponden al producto product_id = 1 y utilizan card.


In [330]:
# Pagos negativos vs pagos del producto 1 realizados con tarjeta
grupo_producto_tarjeta = df.loc[df["product_id"].astype("string").eq("1") & df["pay_method"].eq("card")].copy()

monto_grupo = pd.to_numeric(grupo_producto_tarjeta["payment_amount"].astype("string").str.replace(",", ".", regex=False), errors="coerce")

grupo_producto_tarjeta["monto_analisis"] = monto_grupo
grupo_producto_tarjeta["es_negativo"] = monto_grupo < 0

print("Total de pagos del producto 1 con tarjeta:", len(grupo_producto_tarjeta))

print("\nDistribución de importes:")
display(grupo_producto_tarjeta["monto_analisis"]
    .value_counts()
    .sort_index()
    .to_frame("cantidad")
)

print("\nComparación por signo del importe:")
display(grupo_producto_tarjeta.groupby("es_negativo")["monto_analisis"]
    .agg(["count", "min", "median", "max", "sum"])
)

Total de pagos del producto 1 con tarjeta: 24256

Distribución de importes:


,cantidad
monto_analisis,
-25.0,6328
9.99,1399
11.99,742
12.5,996
13.99,799
15.99,777
17.99,827
19.99,778
21.99,799



Comparación por signo del importe:


,count,min,median,max,sum
es_negativo,,,,,
False,17928,9.99,27.99,50000.0,3839949.35
True,6328,-25.0,-25.0,-25.0,-158200.0


In [331]:

# Negativos vs estado y moneda
tabla_negativos = pd.crosstab(
    [pagos_negativos["payment_status"], pagos_negativos["currency"]],
    pagos_negativos["payment_amount"],
    margins=True
)

display(tabla_negativos)

payment_amount           -25.00   All
payment_status currency              
paid           EURO         206   206
               USD         6105  6105
paidd          USD           17    17
All                        6328  6328

In [332]:
# Imputación temporal de negativos a positivos
monto_original = pd.to_numeric(df["payment_amount"].astype("string").str.replace(",", ".", regex=False),errors="coerce")

monto_positivo = monto_original.abs()

val_negativo = monto_original < 0

# Antes y después
comparacion_negativos = pd.DataFrame({
    "escenario": [
        "Importes originales",
        "Negativos convertidos a positivos"
    ],
    "cantidad_pagos": [
        int(monto_original.notna().sum()),
        int(monto_positivo.notna().sum())
    ],
    "suma_importes": [
        monto_original.sum(),
        monto_positivo.sum()
    ],
    "importe_minimo": [
        monto_original.min(),
        monto_positivo.min()
    ],
    "importe_maximo": [
        monto_original.max(),
        monto_positivo.max()
    ]
})

display(comparacion_negativos.round(2))

# Efecto de la transformación
print("\nPagos negativos que cambiarían:", int(val_negativo.sum()))
print("Importe de los pagos negativos:", round(monto_original[val_negativo].sum(), 2))
print("Importe de esos pagos después del cambio:", round(monto_positivo[val_negativo].sum(), 2))
print(
    "Aumento total de los importes registrados:",
    round(monto_positivo.sum() - monto_original.sum(), 2)
)


,escenario,cantidad_pagos,suma_importes,importe_minimo,importe_maximo
0,Importes originales,100000,16289443.74,-25.00,50000.0
1,Negativos convertidos a positivos,100000,16605843.74,9.99,50000.0



Pagos negativos que cambiarían: 6328
Importe de los pagos negativos: -158200.0
Importe de esos pagos después del cambio: 158200.0
Aumento total de los importes registrados: 316400.0


##### Comentario
- La suma de los importes negativos es −158,200.00. En una simulación temporal, convertirlos a positivos aumenta el total acumulado en 316,400.00 unidades sin modificar la cantidad de transacciones.
- El patrón aparece en 6,122 registros en USD y 206 en EURO. En cuanto al estado, 6,311 registros tienen paid y 17 tienen paidd.
- Las 6,328 transacciones en negativo permanecerán como un patrón pendiente de validación (pueden representan devoluciones, ajustes u otra operación).

#### Investigación de monedas

In [333]:
distribucion_monedas = pd.DataFrame({
    "cantidad": df["currency"].value_counts(dropna=False),
    "porcentaje (%)": (
        df["currency"].value_counts(normalize=True, dropna=False) * 100
    ).round(2)
})

display(distribucion_monedas)

,cantidad,porcentaje (%)
currency,,
USD,93806,93.81
EURO,4805,4.81
USDT,1389,1.39


In [334]:
# Moneda del pago vs moneda del producto
moneda_producto = dict(zip(productos["product_id"], productos["currency"]))

moneda_esperada = df["product_id"].map(moneda_producto)

comparacion_monedas = pd.crosstab(df["currency"], moneda_esperada, margins=True, margins_name="Total")

display(comparacion_monedas)

# Pagos con monedas que difieren de la del catálogo
moneda_diferente = (
    df["currency"].notna()
    & moneda_esperada.notna()
    & df["currency"].ne(moneda_esperada)
)

print("\nPagos con moneda distinta a la del catálogo:", int(moneda_diferente.sum()))

print("Porcentaje del dataset:", round(moneda_diferente.mean() * 100, 2), "%")


product_id,USD,Total
currency,,
EURO,4805,4805
USD,93806,93806
USDT,1389,1389
Total,100000,100000



Pagos con moneda distinta a la del catálogo: 6194
Porcentaje del dataset: 6.19 %


In [335]:
# Moneda del pago por producto
tabla_moneda_producto = pd.crosstab(
    df["product_id"],
    df["currency"],
    margins=True,
    margins_name="Total"
)

print("Distribución por producto:")
display(tabla_moneda_producto)

# Moneda del pago por método de pago
tabla_moneda_metodo = pd.crosstab(
    df["pay_method"],
    df["currency"],
    margins=True,
    margins_name="Total"
)

print("\nDistribución por método de pago:")
display(tabla_moneda_metodo)


Distribución por producto:


currency,EURO,USD,USDT,Total
product_id,,,,
1,2421,21128,707,24256
2,0,13884,0,13884
3,0,15709,0,15709
4,2384,13807,682,16873
5,0,15655,0,15655
6,0,13623,0,13623
Total,4805,93806,1389,100000



Distribución por método de pago:


currency,EURO,USD,USDT,Total
pay_method,,,,
card,4805,34935,1389,41129
cash,0,29332,0,29332
transfer,0,29539,0,29539
Total,4805,93806,1389,100000


##### Comentario
- Las monedas EURO y USDT aparecen exclusivamente en los productos 1 y 4. Los productos 2, 3, 5 y 6 tienen todas sus transacciones registradas en USD.
- Todas las transacciones en EURO y USDT se realizaron mediante card. Los métodos cash y transfer aparecen exclusivamente en USD.

In [336]:

# Convertir importes a formato numérico
monto_analisis = pd.to_numeric(df["payment_amount"].astype("string").str.replace(",", ".", regex=False), errors="coerce")

df_monedas = df.assign(monto_analisis=monto_analisis)

# Comparar los importes por producto y moneda
resumen_importes_monedas = (
    df_monedas[df_monedas["currency"].isin(["USD", "EURO", "USDT"])]
    .groupby(["product_id", "currency"])["monto_analisis"]
    .agg(["count", "min", "median", "max", "sum"])
    .round(2)
    .reset_index()
)

display(resumen_importes_monedas)

,product_id,currency,count,min,median,max,sum
0,1,EURO,2421,-25.0,27.99,47.99,59648.95
1,1,USD,21128,-25.0,17.99,50000.0,3551899.93
2,1,USDT,707,9.99,27.99,50000.0,70200.47
3,2,USD,13884,10.99,28.99,48.99,397137.08
4,3,USD,15709,9.99,27.99,50000.0,5614826.31
5,4,EURO,2384,10.99,28.99,48.99,70942.28
6,4,USD,13807,10.99,28.99,48.99,395221.42
7,4,USDT,682,10.99,28.99,48.99,20129.71
8,5,USD,15655,9.99,27.99,50000.0,5718430.86
9,6,USD,13623,10.99,28.99,48.99,391006.73


##### Comentario
- Para el producto 1, las tres monedas presentan diferencias en la mediana, y en los valores mínimos y máximos.
- Para el producto 4, las tres monedas tienen la misma mediana de 28.99 y los mismos valores mínimo y máximo.

In [337]:
# Distribución de monedas dentro de cada producto
distribucion_pct_producto = (
    pd.crosstab(
        df["product_id"],
        df["currency"],
        normalize="index"
    ) * 100
).round(2)

display(distribucion_pct_producto)


currency,EURO,USD,USDT
product_id,,,
1,9.98,87.10,2.91
2,0.00,100.00,0.00
3,0.00,100.00,0.00
4,14.13,81.83,4.04
5,0.00,100.00,0.00
6,0.00,100.00,0.00


##### Comentario
- Producto 1, el 12.89% de sus transacciones está registrado en EURO o USDT.
- Producto 4, tiene la mayor proporción de monedas distintas de USD, con un 18.17% de sus transacciones.
- Los productos 2, 3, 5 y 6 presentan transacciones únicamente en USD.

In [338]:
# Distribución porcentual de monedas dentro de cada región
distribucion_moneda_region = (
    pd.crosstab(
        df["region_id"].replace("", pd.NA).fillna("Sin región"),
        df["currency"],
        normalize="index"
    ) * 100
).round(2)

display(distribucion_moneda_region)


currency,EURO,USD,USDT
region_id,,,
1,5.46,92.97,1.57
2,4.88,93.80,1.32
3,5.28,93.21,1.50
4,4.83,93.78,1.39
Sin región,0.28,99.13,0.59


##### Comentario
- La distribución porcentual muestra que USD predomina en todas las regiones, representando entre el 92.97% y el 93.80% de las transacciones.
- EURO representa entre el 4.83% y el 5.46%, mientras que USDT representa entre el 1.32% y el 1.57%.
- No hay una concentración evidente de EURO o USDT en una región específica.

#### Investigación de fechas

In [339]:
columnas_fecha = ["pay_dt", "signup_dt"]

diagnostico_fechas = pd.DataFrame({
    "tipo_original": df[columnas_fecha].dtypes.astype(str),
    "cadenas_vacias": {
        col: df[col].eq("").sum()
        for col in columnas_fecha
    },
    "nulos_reales": df[columnas_fecha].isna().sum(),
    "valores_unicos": df[columnas_fecha].nunique()
})

display(diagnostico_fechas)

,tipo_original,cadenas_vacias,nulos_reales,valores_unicos
pay_dt,str,0,0,91859
signup_dt,str,6122,0,732


In [340]:
# Conversión temporal 
pay_dt_temporal = pd.to_datetime(df["pay_dt"], format="%Y-%m-%d %H:%M:%S", errors="coerce")

signup_dt_temporal = pd.to_datetime(df["signup_dt"], format="%Y-%m-%d", errors="coerce")

resumen_fechas = pd.DataFrame({
    "columna": ["pay_dt", "signup_dt"],
    "registros_originales": [len(df), len(df)],
    "cadenas_vacias": [
        df["pay_dt"].eq("").sum(),
        df["signup_dt"].eq("").sum()
    ],
    "fechas_no_convertibles": [
        (df["pay_dt"].ne("") & pay_dt_temporal.isna()).sum(),
        (df["signup_dt"].ne("") & signup_dt_temporal.isna()).sum()
    ],
    "fechas_convertibles": [
        pay_dt_temporal.notna().sum(),
        signup_dt_temporal.notna().sum()
    ]
})

display(resumen_fechas)


,columna,registros_originales,cadenas_vacias,fechas_no_convertibles,fechas_convertibles
0,pay_dt,100000,0,5549,94451
1,signup_dt,100000,6122,57,93821


In [341]:
# Valores originales no vacíos que no se pudieron convertir
fechas_pago_invalidas = df.loc[df["pay_dt"].ne("") & pay_dt_temporal.isna(), "pay_dt"].value_counts().to_frame("cantidad")

fechas_registro_invalidas = df.loc[df["signup_dt"].ne("") & signup_dt_temporal.isna(), "signup_dt"].value_counts().to_frame("cantidad")

print("Valores inválidos en pay_dt:")
display(fechas_pago_invalidas)

print("\nValores inválidos en signup_dt:")
display(fechas_registro_invalidas)

Valores inválidos en pay_dt:


,cantidad
pay_dt,
2024-02-30,5549



Valores inválidos en signup_dt:


,cantidad
signup_dt,
2024-13-40,57


##### Comentario
- pay_dt, las 5,549 fechas inválidas siguen el mismo patrón, 2024-02-30. El 30 de febrero no existe.
- signup_dt, hay 57 fechas inválidas con el patrón 2024-13-40, que contiene un mes y un día imposibles. Además, hay 6,122 registros sin fecha de registro.
- Ambas columnas están almacenadas como texto (str).
- No hay nulos reales en las columnas originales; las ausencias de signup_dt están representadas por cadenas vacías.

In [342]:
# Imputación temporal fechas inválidas
comparacion_fechas = pd.DataFrame({
    "escenario": [
        "pay_dt original",
        "pay_dt convertido",
        "signup_dt original",
        "signup_dt convertido"
    ],
    "registros": [
        len(df),
        len(pay_dt_temporal),
        len(df),
        len(signup_dt_temporal)
    ],
    "fechas_validas": [
        df["pay_dt"].ne("").sum() - fechas_pago_invalidas["cantidad"].sum(),
        pay_dt_temporal.notna().sum(),
        df["signup_dt"].ne("").sum() - fechas_registro_invalidas["cantidad"].sum(),
        signup_dt_temporal.notna().sum()
    ],
    "fechas_invalidas_o_ausentes": [
        df["pay_dt"].eq("").sum() + fechas_pago_invalidas["cantidad"].sum(),
        pay_dt_temporal.isna().sum(),
        df["signup_dt"].eq("").sum() + fechas_registro_invalidas["cantidad"].sum(),
        signup_dt_temporal.isna().sum()
    ]
})

display(comparacion_fechas)

,escenario,registros,fechas_validas,fechas_invalidas_o_ausentes
0,pay_dt original,100000,94451,5549
1,pay_dt convertido,100000,94451,5549
2,signup_dt original,100000,93821,6179
3,signup_dt convertido,100000,93821,6179


##### Comentario
- Al convertir las columnas "pay_dt" y "signup_dt" a tipos de fecha, los registros se interpretan temporalmente como NaT, sin eliminar transacciones ni modificar el dataset.

#### Investigación de payment_id repetidos y US$50,000.

In [343]:
# Pagos de exactamente 50,000
monto_analisis_50000 = pd.to_numeric(
    df["payment_amount"]
    .astype("string")
    .str.replace(",", ".", regex=False),
    errors="coerce"
)

pagos_50000 = df.loc[monto_analisis_50000.eq(50000)].copy()
pagos_50000["monto_analisis_50000"] = monto_analisis_50000.loc[pagos_50000.index]

# payment_id repetidos en el dataset completo
ids_repetidos = df["payment_id"].duplicated(keep=False)

pagos_50000_repetidos = pagos_50000.loc[
    pagos_50000["payment_id"].isin(df.loc[ids_repetidos, "payment_id"])
]

print("Pagos de exactamente 50,000:", len(pagos_50000))
print("Pagos de 50,000 con payment_id repetido:",
    len(pagos_50000_repetidos)
)
print("Payment_id distintos entre esos pagos:",
    pagos_50000_repetidos["payment_id"].nunique()
)
print("Porcentaje de pagos de 50,000 con ID repetido:",
    round(len(pagos_50000_repetidos) / len(pagos_50000) * 100, 2),
    "%"
)


Pagos de exactamente 50,000: 277
Pagos de 50,000 con payment_id repetido: 221
Payment_id distintos entre esos pagos: 221
Porcentaje de pagos de 50,000 con ID repetido: 79.78 %


In [344]:
# ids de los pagos de 50,000 que están repetidos
ids_50000_repetidos = pagos_50000_repetidos["payment_id"].unique()

# Transacciones asociadas a esos ids
transacciones_relacionadas = df.loc[df["payment_id"].isin(ids_50000_repetidos)].copy()

transacciones_relacionadas["monto_analisis"] = monto_analisis.loc[
transacciones_relacionadas.index
]

transacciones_relacionadas = transacciones_relacionadas.sort_values(
    ["payment_id", "monto_analisis"]
)

print("Transacciones relacionadas:", len(transacciones_relacionadas))

display(
    transacciones_relacionadas[
        [
            "payment_id",
            "customer_id",
            "product_id",
            "payment_amount",
            "payment_status",
            "currency",
            "pay_method",
            "pay_dt"
        ]
    ]
)

Transacciones relacionadas: 442


,payment_id,customer_id,product_id,payment_amount,payment_status,currency,pay_method,pay_dt
1641,5000001,5000002,4,10.99,paid,USD,card,2024-02-12 13:46:41
57011,5000001,5000001,3,50000.00,paid,USD,cash,2024-02-12 13:46:40
57269,5020001,5020002,6,10.99,paid,USD,cash,2024-02-12 19:20:01
63621,5020001,5020001,5,50000.00,paid,USD,transfer,2024-02-12 19:20:00
24560,5060001,5060002,4,10.99,paid,USD,card,2024-02-13 06:26:41
...,...,...,...,...,...,...,...,...
56156,9940001,109940000,5,50000.00,paid,USD,transfer,2024-02-23 10:53:20
92246,9960001,9960002,2,10.99,paid,USD,transfer,2024-02-23 16:26:41
27867,9960001,9960001,1,50000.00,paid,USD,card,2024-02-23 16:26:40
81529,9980001,9980002,4,10.99,paid,USD,card,2024-02-23 22:00:01


In [345]:
# Transacciones relacionadas por payment_id
resumen_ids = (
    transacciones_relacionadas
    .groupby("payment_id")
    .agg(
        cantidad_filas=("payment_id", "size"),
        clientes_distintos=("customer_id", "nunique"),
        productos_distintos=("product_id", "nunique"),
        importes_distintos=("monto_analisis", "nunique"),
        estados_distintos=("payment_status", "nunique"),
        monedas_distintas=("currency", "nunique")
    )
    .reset_index()
)

display(resumen_ids)

print("\nDistribución de filas por payment_id:")
display(
    resumen_ids["cantidad_filas"]
    .value_counts()
    .sort_index()
    .to_frame("cantidad_de_payment_id")
)

,payment_id,cantidad_filas,clientes_distintos,productos_distintos,importes_distintos,estados_distintos,monedas_distintas
0,5000001,2,2,2,2,1,1
1,5020001,2,2,2,2,1,1
2,5060001,2,2,2,2,1,1
3,5080001,2,2,2,2,1,1
4,5100001,2,2,2,2,1,1
...,...,...,...,...,...,...,...
216,9880001,2,2,2,2,1,1
217,9920001,2,2,2,2,1,1
218,9940001,2,2,2,2,1,1
219,9960001,2,2,2,2,1,1



Distribución de filas por payment_id:


,cantidad_de_payment_id
cantidad_filas,
2,221


##### Comentario
- Se identificaron 277 transacciones con un importe de exactamente 50,000. De estas, 221 (79.78%) tienen un payment_id que aparece repetido en el dataset. Al recuperar todas las transacciones asociadas a esos ids, se obtuvieron 442 registros, correspondientes a 221 pares.

In [346]:
# payment_id vs customer_id en las 442 transacciones relacionadas
pares_cliente_id = transacciones_relacionadas.copy()

# Convertir ambos ids a números
pares_cliente_id["payment_id_num"] = pd.to_numeric(pares_cliente_id["payment_id"], errors="coerce")

pares_cliente_id["customer_id_num"] = pd.to_numeric(pares_cliente_id["customer_id"], errors="coerce")

# Dif.
pares_cliente_id["diferencia_id_cliente"] = (pares_cliente_id["customer_id_num"] - pares_cliente_id["payment_id_num"])

print("Distribución de la diferencia customer_id - payment_id:")
display(
    pares_cliente_id["diferencia_id_cliente"]
    .value_counts(dropna=False)
    .sort_index()
    .to_frame("cantidad")
)

# Comparar igualdad
pares_cliente_id["id_igual_cliente"] = (pares_cliente_id["customer_id_num"] == pares_cliente_id["payment_id_num"])

print("\nCoincidencia entre payment_id y customer_id:")
display(
    pares_cliente_id["id_igual_cliente"]
    .value_counts(dropna=False)
    .to_frame("cantidad")
)

Distribución de la diferencia customer_id - payment_id:


,cantidad
diferencia_id_cliente,
0,189
1,221
99999999,32



Coincidencia entre payment_id y customer_id:


,cantidad
id_igual_cliente,
False,253
True,189


##### Comentario
- 221 registros presentan una diferencia de exactamente 1.
- 189 registros tienen ambos identificadores iguales. 
- 32 registros presentan una diferencia de 99,999,999. Conviene examinar esos casos individualmente para comprobar si existe un patrón específico.

In [347]:
# Casos atípicos
casos_diferencia_extrema = pares_cliente_id.loc[pares_cliente_id["diferencia_id_cliente"].eq(99999999)].copy()

display(
    casos_diferencia_extrema[
        [
            "payment_id",
            "customer_id",
            "product_id",
            "payment_amount",
            "payment_status",
            "currency",
            "pay_method"
        ]
    ]
)

,payment_id,customer_id,product_id,payment_amount,payment_status,currency,pay_method
77664,5180001,105180000,3,50000.00,paid,USD,cash
70535,5320001,105320000,5,50000.00,paid,USD,transfer
71254,5460001,105460000,1,50000.00,paid,USD,card
69562,5600001,105600000,3,50000.00,paid,USD,cash
9575,5740001,105740000,5,50000.00,paid,USD,transfer
35129,5880001,105880000,1,50000.00,paid,USD,card
69483,6020001,106020000,3,50000.00,paid,USD,cash
54566,6160001,106160000,5,50000.00,paid,USD,transfer
87903,6440001,106440000,3,50000.00,paid,USD,cash
47342,6580001,106580000,5,50000.00,paid,USD,transfer


In [348]:
print("Cantidad de registros:", len(casos_diferencia_extrema))
print("payment_id distintos:",
    casos_diferencia_extrema["payment_id"].nunique()
)
print("customer_id distintos:",
    casos_diferencia_extrema["customer_id"].nunique()
)

print("\nDistribución por producto:")
display(
    casos_diferencia_extrema["product_id"]
    .value_counts()
    .sort_index()
    .to_frame("cantidad")
)

print("\nDistribución por método de pago:")
display(
    casos_diferencia_extrema["pay_method"]
    .value_counts()
    .to_frame("cantidad")
)

print("\nDistribución por estado:")
display(
    casos_diferencia_extrema["payment_status"]
    .value_counts(dropna=False)
    .to_frame("cantidad")
)


Cantidad de registros: 32
payment_id distintos: 32
customer_id distintos: 32

Distribución por producto:


,cantidad
product_id,
1,8
3,12
5,12



Distribución por método de pago:


,cantidad
pay_method,
cash,12
transfer,12
card,8



Distribución por estado:


,cantidad
payment_status,
paid,32


In [349]:
# Transacciones de los 32 clientes atípicos
clientes_atipicos = casos_diferencia_extrema["customer_id"].unique()

transacciones_clientes_atipicos = df.loc[df["customer_id"].isin(clientes_atipicos)].copy()

print("Transacciones asociadas a los 32 clientes:",
    len(transacciones_clientes_atipicos)
)

print("Clientes distintos encontrados:",
    transacciones_clientes_atipicos["customer_id"].nunique()
)

resumen_clientes_atipicos = (
    transacciones_clientes_atipicos
    .groupby("customer_id")
    .agg(
        cantidad_pagos=("payment_id", "size"),
        payment_ids_distintos=("payment_id", "nunique"),
        productos_distintos=("product_id", "nunique"),
        importes_distintos=("payment_amount", "nunique")
    )
    .reset_index()
)

display(resumen_clientes_atipicos)


Transacciones asociadas a los 32 clientes: 58
Clientes distintos encontrados: 32


,customer_id,cantidad_pagos,payment_ids_distintos,productos_distintos,importes_distintos
0,105180000,2,2,1,1
1,105320000,1,1,1,1
2,105460000,1,1,1,1
3,105600000,2,2,1,1
4,105740000,2,2,1,1
5,105880000,2,2,1,1
6,106020000,2,2,1,1
7,106160000,2,2,1,1
8,106440000,2,2,1,1
9,106580000,2,2,1,1


In [350]:
# Examinar los importes y productos de los 32 clientes
detalle_clientes_atipicos = (
    transacciones_clientes_atipicos
    .groupby(
        ["customer_id", "product_id", "payment_amount"],
        dropna=False
    )
    .agg(
        cantidad_transacciones=("payment_id", "size"),
        payment_ids=("payment_id", lambda x: list(x))
    )
    .reset_index()
)

print("Distribución de importes:")
display(
    transacciones_clientes_atipicos["payment_amount"]
    .value_counts(dropna=False)
    .to_frame("cantidad")
)

print("\nDistribución por producto e importe:")
display(
    detalle_clientes_atipicos.sort_values(
        ["product_id", "payment_amount"]
    )
)


Distribución de importes:


,cantidad
payment_amount,
50000.00,58



Distribución por producto e importe:


,customer_id,product_id,payment_amount,cantidad_transacciones,payment_ids
2,105460000,1,50000.00,1,[5460001]
5,105880000,1,50000.00,2,"[5880001, 5880000]"
10,106720000,1,50000.00,1,[6720001]
13,107140000,1,50000.00,2,"[7140001, 7140000]"
18,107980000,1,50000.00,1,[7980001]
21,108400000,1,50000.00,2,"[8400001, 8400000]"
26,109240000,1,50000.00,2,"[9240000, 9240001]"
29,109660000,1,50000.00,2,"[9660000, 9660001]"
0,105180000,3,50000.00,2,"[5180000, 5180001]"
3,105600000,3,50000.00,2,"[5600001, 5600000]"


In [351]:
# Comprobar si los payment_id relacionados aparecen con otros importes
ids_clientes_atipicos = transacciones_clientes_atipicos["payment_id"].unique()

revision_ids_atipicos = df.loc[
    df["payment_id"].isin(ids_clientes_atipicos)
].copy()

revision_ids_atipicos["monto_analisis"] = pd.to_numeric(
    revision_ids_atipicos["payment_amount"]
    .astype("string")
    .str.replace(",", ".", regex=False),
    errors="coerce"
)

resumen_ids_atipicos = (
    revision_ids_atipicos
    .groupby("payment_id")
    .agg(
        cantidad_filas=("payment_id", "size"),
        importes_distintos=("monto_analisis", "nunique"),
        clientes_distintos=("customer_id", "nunique"),
        productos_distintos=("product_id", "nunique")
    )
    .reset_index()
)

display(
    resumen_ids_atipicos[
        resumen_ids_atipicos["cantidad_filas"] > 1
    ]
)

,payment_id,cantidad_filas,importes_distintos,clientes_distintos,productos_distintos
1,5180001,2,2,2,2
2,5320001,2,2,2,2
3,5460001,2,2,2,2
5,5600001,2,2,2,2
7,5740001,2,2,2,2
9,5880001,2,2,2,2
11,6020001,2,2,2,2
13,6160001,2,2,2,2
15,6440001,2,2,2,2
17,6580001,2,2,2,2


##### Comentario
- Se identificaron 277 pagos de exactamente 50,000.
- De estos, 221 (79.78%) tienen un payment_id repetido.
- En los 32 casos con una diferencia de 99,999,999 entre customer_id y payment_id, los clientes tienen transacciones repetidas por 50,000, con el mismo producto e importe para cada cliente.

In [352]:
# Patrón numérico de los identificadores atípicos
secuencia_ids = casos_diferencia_extrema[
    ["payment_id", "customer_id", "product_id", "payment_amount"]
].copy()

secuencia_ids["payment_id_num"] = pd.to_numeric(
    secuencia_ids["payment_id"], errors="coerce"
)

secuencia_ids["customer_id_num"] = pd.to_numeric(
    secuencia_ids["customer_id"], errors="coerce"
)

secuencia_ids = secuencia_ids.sort_values("payment_id_num")

# Diferencia entre un identificador y el siguiente
secuencia_ids["diferencia_payment_id"] = (
    secuencia_ids["payment_id_num"].diff()
)

print("Identificadores ordenados:")
display(secuencia_ids)

print("\nFrecuencia de las diferencias entre payment_id:")
display(
    secuencia_ids["diferencia_payment_id"]
    .value_counts()
    .sort_index()
    .to_frame("cantidad")
)

print("\nDiferencia entre customer_id y payment_id:")
display(
    (
        secuencia_ids["customer_id_num"]
        - secuencia_ids["payment_id_num"]
    )
    .value_counts()
    .sort_index()
    .to_frame("cantidad")
)


Identificadores ordenados:


,payment_id,customer_id,product_id,payment_amount,payment_id_num,customer_id_num,diferencia_payment_id
77664,5180001,105180000,3,50000.00,5180001,105180000,NaN
70535,5320001,105320000,5,50000.00,5320001,105320000,140000.0
71254,5460001,105460000,1,50000.00,5460001,105460000,140000.0
69562,5600001,105600000,3,50000.00,5600001,105600000,140000.0
9575,5740001,105740000,5,50000.00,5740001,105740000,140000.0
35129,5880001,105880000,1,50000.00,5880001,105880000,140000.0
69483,6020001,106020000,3,50000.00,6020001,106020000,140000.0
54566,6160001,106160000,5,50000.00,6160001,106160000,140000.0
87903,6440001,106440000,3,50000.00,6440001,106440000,280000.0
47342,6580001,106580000,5,50000.00,6580001,106580000,140000.0



Frecuencia de las diferencias entre payment_id:


,cantidad
diferencia_payment_id,
140000.0,28
280000.0,3



Diferencia entre customer_id y payment_id:


,cantidad
99999999,32


##### Comentario
- Los 32 casos corresponden a clientes distintos y presentan el mismo producto y el mismo importe de 50,000 en sus transacciones. 
- Los "payment_id" siguen una secuencia numérica estructurada: 28 de las 31 diferencias consecutivas son de 140,000 y las otras 3 son de 280,000.

In [353]:
# Comparar las columnas de los registros con payment_id repetido
# dentro de las transacciones relacionadas con pagos de 50,000
columnas_comparar = [
    columna for columna in transacciones_relacionadas.columns
    if columna not in ["monto_analisis"]
]

diferencias_por_id = (
    transacciones_relacionadas
    .groupby("payment_id")[columnas_comparar]
    .nunique(dropna=False)
)

# Contar en cuántos pares cambia cada columna
resumen_diferencias = pd.DataFrame({
    "payment_id_con_diferencias": (
        diferencias_por_id.gt(1).sum()
    ),
    "total_payment_id": len(diferencias_por_id)
})

resumen_diferencias["porcentaje_con_diferencias"] = (
    resumen_diferencias["payment_id_con_diferencias"]
    / resumen_diferencias["total_payment_id"] * 100
).round(2)

display(resumen_diferencias)

,payment_id_con_diferencias,total_payment_id,porcentaje_con_diferencias
payment_id,0,221,0.00
customer_id,221,221,100.00
product_id,221,221,100.00
payment_amount,221,221,100.00
payment_status,0,221,0.00
currency,1,221,0.45
pay_method,221,221,100.00
pay_dt,221,221,100.00
region_id,221,221,100.00
status_customer,32,221,14.48


##### Comentario
- En los 221 pares relacionados con pagos de 50,000, se identificaron diferencias en customer_id, product_id, payment_amount, pay_method, pay_dt y region_id en el 100% de los pares. status_customer presentó diferencias en 32 pares (14.48%), mientras que payment_status fue igual en todos los pares.
- Este patrón demuestra que los registros con el mismo payment_id contienen información transaccional diferente y no son duplicados completos. 

### Modificaciones luego de revisión

In [354]:
montos = pd.to_numeric(
    df["payment_amount"].astype("string").str.replace(
        ",", ".", regex=False
    ),
    errors="coerce"
)

pagos_50000 = montos.eq(50000)

comparacion_50000 = pd.DataFrame({
    "escenario": ["Todos los pagos", "Sin pagos de 50,000"],
    "cantidad_pagos": [
        len(df),
        int((~pagos_50000).sum())
    ],
    "total_importes": [
        montos.sum(),
        montos.loc[~pagos_50000].sum()
    ]
})

display(comparacion_50000)

,escenario,cantidad_pagos,total_importes
0,Todos los pagos,100000,16289443.74
1,"Sin pagos de 50,000",99723,2439443.74


In [355]:
print("Suma original:", montos.sum().round())
print("Suma de los pagos de 50,000:", montos[pagos_50000].sum())
print("Suma sin pagos de 50,000:", montos[~pagos_50000].sum())

print("Comprobación:", montos[pagos_50000].sum() + montos[~pagos_50000].sum())

Suma original: 16289444.0
Suma de los pagos de 50,000: 13850000.0
Suma sin pagos de 50,000: 2439443.74
Comprobación: 16289443.74


##### Comentario
- Al excluir los 277 pagos de 50000, el número de registros pasa de 100,000 a 99,723 y el total monetario disminuye considerablemente.

In [356]:
# Crear dos escenarios temporales sin modificar df.
df_todos = df.copy()
df_sin_50000 = df.loc[~pagos_50000].copy()

# Marcar pagos fallidos y contracargos.
df_todos["es_fallido"] = df_todos["payment_status"].isin(
    ["failed", "chargeback"]
)

df_sin_50000["es_fallido"] = df_sin_50000["payment_status"].isin(
    ["failed", "chargeback"]
)

# Calcular total de pagos y tasa de fallos por región.
def tasa_fallos_region(datos):
    resultado = (
        datos.groupby("region_id", dropna=False)
        .agg(
            total_pagos=("payment_id", "size"),
            pagos_fallidos=("es_fallido", "sum")
        )
    )

    resultado["tasa_fallos_pct"] = (
        resultado["pagos_fallidos"] / resultado["total_pagos"] * 100
    ).round(2)

    return resultado.sort_values(
        "tasa_fallos_pct", ascending=False
    )

print("TODOS LOS PAGOS")
display(tasa_fallos_region(df_todos))

print("SIN PAGOS DE 50,000")
display(tasa_fallos_region(df_sin_50000))


TODOS LOS PAGOS


,total_pagos,pagos_fallidos,tasa_fallos_pct
region_id,,,
3,22371,3310,14.80
1,22009,3237,14.71
4,24833,3226,12.99
2,24665,2466,10.00
,6122,0,0.00


SIN PAGOS DE 50,000


,total_pagos,pagos_fallidos,tasa_fallos_pct
region_id,,,
3,22371,3310,14.80
1,22009,3237,14.71
4,24833,3226,12.99
2,24474,2466,10.08
,6036,0,0.00


### Comentario
- Al comparar las tasas de pagos fallidos y contracargos con todos los registros y excluyendo temporalmente los 277 pagos de 50,000. La región 3 mantuvo la tasa más alta, con 14.80%, seguida de la región 1, con 14.71%. La región 2 aumentó ligeramente de 10.00% a 10.08%, debido a la reducción del total de pagos de esa región.

In [357]:
def tasa_fallos_riesgo(datos):
    resultado = (
        datos.groupby("risk_segment", dropna=False)
        .agg(
            total_pagos=("payment_id", "size"),
            pagos_fallidos=("es_fallido", "sum")
        )
    )

    resultado["tasa_fallos_pct"] = (
        resultado["pagos_fallidos"] / resultado["total_pagos"] * 100
    ).round(2)

    return resultado.sort_values("tasa_fallos_pct", ascending=False)


print("TODOS LOS PAGOS")
display(tasa_fallos_riesgo(df_todos))

print("SIN PAGOS DE 50,000")
display(tasa_fallos_riesgo(df_sin_50000))

TODOS LOS PAGOS


,total_pagos,pagos_fallidos,tasa_fallos_pct
risk_segment,,,
High,13344,2484,18.62
Medium,21850,4010,18.35
Low,58684,5745,9.79
,6122,0,0.00


SIN PAGOS DE 50,000


,total_pagos,pagos_fallidos,tasa_fallos_pct
risk_segment,,,
High,13344,2484,18.62
Medium,21850,4010,18.35
Low,58493,5745,9.82
,6036,0,0.00


### Comentario
- Al comparar las tasas de pagos fallidos y contracargos con todos los registros y excluyendo los pagos de 50,000. El segmento High mantuvo la tasa más alta, con 18.62%, seguido de Medium, con 18.35%, y Low, con 9.82%.
- La exclusión de los importes extremos no modificó el orden de los segmentos y solo produjo un aumento de 0.03 puntos en la tasa del segmento Low. 

In [358]:
df_investigacion = df.copy()

# Normalizar la clave regional solo en la copia
df_investigacion["region_id"] = pd.to_numeric(
    df_investigacion["region_id"],
    errors="coerce"
).astype("Int64")

# Identificar los pagos de 50,000.00
pagos_50000 = df_investigacion.loc[
    df_investigacion["payment_amount"]
    .astype("string")
    .str.strip()
    .eq("50000.00"),
    [
        "payment_id",
        "customer_id",
        "product_id",
        "payment_amount",
        "currency",
        "region_id",
        "pay_dt"
    ]
].copy()

print("Cantidad de pagos:", len(pagos_50000))
print(pagos_50000.columns.tolist())

Cantidad de pagos: 277
['payment_id', 'customer_id', 'product_id', 'payment_amount', 'currency', 'region_id', 'pay_dt']


In [359]:
# Preparar las columnas de la tabla de regiones
regiones_limpias = regiones[
    ["region_id", "region_name", "country"]
].copy()

regiones_limpias["region_id"] = pd.to_numeric(
    regiones_limpias["region_id"],
    errors="coerce"
).astype("Int64")

# Relacionar los pagos de 50,000 con su región
pagos_con_region = pagos_50000.merge(
    regiones_limpias,
    on="region_id",
    how="left",
    validate="many_to_one",
    indicator=True
)

print("Total de pagos investigados:", len(pagos_con_region))
print("\nCorrespondencia regional:")
print(pagos_con_region["_merge"].value_counts())


Total de pagos investigados: 277

Correspondencia regional:
_merge
both          191
left_only      86
right_only      0
Name: count, dtype: int64


#### Comentario
- El cruce regional permite relacionar 191 de los 277 pagos de 50,000 con una región identificada, mientras que 86 registros no encontraron correspondencia. La falta de correspondencia debe investigarse.

In [360]:
# 1. Preparar los pagos sin columnas regionales anteriores
# para evitar duplicados de nombres al hacer el merge.

pagos_base = df_investigacion[
    [
        "payment_id",
        "product_id",
        "payment_amount",
        "currency",
        "region_id"
    ]
].copy()

# Convertir el importe a numérico sin modificar el DataFrame original.
pagos_base["payment_amount_num"] = pd.to_numeric(
    pagos_base["payment_amount"]
    .astype("string")
    .str.strip()
    .str.replace(",", ".", regex=False),
    errors="coerce"
)

# 2. Seleccionar los registros cuyo importe es exactamente 50,000.
pagos_50000 = pagos_base.loc[
    pagos_base["payment_amount_num"].eq(50000)
].copy()

print("Cantidad de pagos de 50,000:", len(pagos_50000))

# 3. Preparar la tabla de regiones y verificar su clave.
regiones_limpias = regiones[
    ["region_id", "region_name", "country"]
].copy()

regiones_limpias["region_id"] = pd.to_numeric(
    regiones_limpias["region_id"],
    errors="coerce"
).astype("Int64")

pagos_50000["region_id"] = pd.to_numeric(
    pagos_50000["region_id"],
    errors="coerce"
).astype("Int64")

# 4. Relacionar cada pago con su región.
pagos_con_region = pagos_50000.merge(
    regiones_limpias,
    on="region_id",
    how="left",
    validate="many_to_one",
    indicator=True
)

print("\nResultado del cruce:")
print(pagos_con_region["_merge"].value_counts())

# 5. Contar los pagos por producto, región y moneda.
resumen_c = (
    pagos_con_region
    .groupby(
        ["product_id", "region_name", "country", "currency"],
        dropna=False
    )
    .size()
    .rename("cantidad_pagos")
    .reset_index()
    .sort_values("cantidad_pagos", ascending=False)
)

print("\nDistribución de los pagos de 50,000:")
print(resumen_c.to_string(index=False))


Cantidad de pagos de 50,000: 277

Resultado del cruce:
_merge
both          191
left_only      86
right_only      0
Name: count, dtype: int64

Distribución de los pagos de 50,000:
 product_id region_name country currency  cantidad_pagos
          3      Andina      CO      USD              72
          5      Andina      CO      USD              72
          1      Andina      CO      USD              47
          5         NaN     NaN      USD              34
          3         NaN     NaN      USD              32
          1         NaN     NaN      USD              19
          1         NaN     NaN     USDT               1


#### Comentario
- Los pagos de 50,000 identificados en Andina, Colombia, se concentran en los productos 3 y 5, con 72 registros cada uno, seguidos del producto 1, con 47 registros. 
- Existen 86 pagos sin correspondencia regional, incluidos 85 etiquetados como USD y uno como USDT.
- La concentración en Colombia constituye una asociación observada entre el importe y la región.

#### Comentario general - pagos de 50,000
- Se identificaron 277 registros con un importe de 50,000. De estos, 191 están asociados con la región Andina, Colombia (68.95%), mientras que 86 no tienen correspondencia con la tabla de regiones (31.05%).
- Dentro de los registros asociados con Colombia, los productos 3 y 5 presentan la mayor frecuencia, con 72 pagos cada uno, seguidos del producto 1, con 47 pagos.
- Los 277 registros están etiquetados como USD o USDT: 276 corresponden a USD y uno a USDT. No se observa una etiqueta de moneda COP en este grupo.
- La concentración de pagos de 50,000 en Colombia constituye una asociación observada entre el importe y la región, pero no permite determinar por sí sola si hubo un error de conversión monetaria.
- La ausencia de correspondencia regional en 86 registros limita la interpretación geográfica del conjunto.

##### Decisión de análisis
- Se conservarán los datos originales y los 277 registros se marcarán como posibles anomalías para evaluar su impacto en los resultados. 
- La exclusión definitiva queda pendiente de validación con el negocio.

In [361]:
import importlib
import analisis

importlib.reload(analisis)


df_limpio = limpiar(df)

df_transformado = analisis.transformar(
    df_limpio,
    productos,
    regiones
)


print("\n" + "Dimensiones:", df_transformado.shape)
print("Filas:", len(df_transformado))
print("Regiones sin asociar:", df_transformado["region_name"].isna().sum())



INVESTIGACIÓN Y LIMPIEZA
*IMPORTES CON COMA
Valores con coma antes: 6019
Valores con coma después: 0
Valores que no pudieron convertirse: 0
----------------------------------------

*IMPUTACIÓN PAIDD
'paidd' antes: 6030
'paid' antes: 73427
'paidd' después: 0
'paid' después: 79457
----------------------------------------

*IMPUTACIÓN MONEDAS
EURO antes: 4805
USDT antes: 1389
USD antes: 93806
EURO después: 0
USDT después: 0
USD después: 100000
----------------------------------------

*IMPUTACIÓN FECHAS
pay_dt inválidas antes de limpiar: 5549
pay_dt NaT después: 5549

signup_dt inválidas antes de limpiar: 57
signup_dt NaT después: 6179
----------------------------------------

*IMPUTACIÓN CLIENTES SIN DATOS
Filas sin datos de perfil antes: 6122
Filas marcadas sin datos de perfil: 6122
region_id faltantes después: 6122
----------------------------------------

*IMPUTACIÓN MONTOS NEGATIVOS
Importes negativos marcados: 6328
----------------------------------------

Filas completamente dupl

In [362]:
print(
    df_transformado[
        [
            "mes",
            "dia_semana",
            "rango_monto",
            "es_fallido"
        ]
    ].head()
)

print("\nEstados de es_fallido:")
print(df_transformado["es_fallido"].value_counts(dropna=False))

print("\nRangos de monto:")
print(df_transformado["rango_monto"].value_counts(dropna=False))

   mes dia_semana rango_monto  es_fallido
0  2.0   Thursday       medio       False
1  2.0     Friday        alto       False
2  2.0     Friday        bajo       False
3  2.0     Sunday        bajo       False
4  2.0   Thursday        alto       False

Estados de es_fallido:
es_fallido
False    87761
True     12239
Name: count, dtype: int64

Rangos de monto:
rango_monto
bajo     41592
alto     28897
medio    23183
NaN       6328
Name: count, dtype: int64


#### Comentario 
- La variable es_fallido identifica 12,239 pagos como fallidos o contracargos, mientras que 87,761 pagos no pertenecen a estas categorías.
- Los importes se clasificaron en tres rangos: bajo (47,920), medio (23,183) y alto (28,897), para comparar el comportamiento de los pagos según su monto.

In [363]:
print(df_transformado["payment_status"].value_counts(dropna=False))
print(" ")
print(
    pd.crosstab(
        df_transformado["payment_status"],
        df_transformado["es_fallido"]
    )
)

payment_status
paid          79457
failed         9792
pending        8304
chargeback     2447
Name: count, dtype: int64
 
es_fallido      False  True 
payment_status              
chargeback          0   2447
failed              0   9792
paid            79457      0
pending          8304      0


#### Comentario 
- Se plantea nuevamente la distribución de payment_status, se utiliza una tabla cruzada para comprobar si la columna es_fallido clasifica correctamente los estados.
- La clasificación marca como True los pagos con estado "failed" y "chargeback", y como False los demás estados.

In [364]:
import importlib
import analisis

# Recargar analisis.py para aplicar los cambios guardados
importlib.reload(analisis)

# 1. Análisis de la tasa de fallos por región
print("=" * 60)
print("1. TASA DE FALLOS POR REGIÓN")
print("=" * 60)

resultado_region = analisis.analizar_region(df_transformado)
display(resultado_region)


# 2. Análisis de importes por producto y método de pago
print("\n" + "=" * 60)
print("2. IMPORTES POR PRODUCTO Y MÉTODO DE PAGO")
print("=" * 60)

resultado_producto_con, resultado_producto_sin = (
    analisis.analizar_producto_metodo(df_transformado)
)

print("\nCon todos los pagos, incluidos los de 50,000:")
display(resultado_producto_con)

print("\nSin los pagos de 50,000:")
display(resultado_producto_sin)


# Comparación de los totales
print("\nComparación de importes totales:")

comparacion_totales = (
    resultado_producto_con[["Total"]]
    .rename(columns={"Total": "total_con_50000"})
    .join(
        resultado_producto_sin[["Total"]].rename(
            columns={"Total": "total_sin_50000"}
        ),
        how="outer"
    )
    .fillna(0)
)

comparacion_totales["diferencia"] = (
    comparacion_totales["total_con_50000"]
    - comparacion_totales["total_sin_50000"]
)

display(comparacion_totales)


# 3. Análisis de la tasa de fallos por segmento de riesgo
print("\n" + "=" * 60)
print("3. TASA DE FALLOS POR SEGMENTO DE RIESGO")
print("=" * 60)

resultado_riesgo = analisis.analizar_riesgo(df_transformado)
display(resultado_riesgo)


1. TASA DE FALLOS POR REGIÓN


,total_pagos,pagos_fallidos,tasa_fallos_pct
region_name,,,
Europa,22371,3310,14.80
Caribe,22009,3237,14.71
Norteamérica,24833,3226,12.99
Andina,24665,2466,10.00
NaN,6122,0,0.00



2. IMPORTES POR PRODUCTO Y MÉTODO DE PAGO

RESUMEN DE SENSIBILIDAD
Método líder con todos los pagos: transfer = US$ 6,115,567.94
Método líder sin pagos de 50,000: card = US$ 818,042.76
Producto líder con todos los pagos: Card = US$ 6,115,567.94
Producto líder sin pagos de 50,000: Loan = US$ 901,119.72

Con todos los pagos, incluidos los de 50,000:


pay_method,card,cash,transfer,Total
product_type,,,,
Card,0.0,0.0,6115567.94,6115567.94
Loan,486293.41,5614826.31,0.0,6101119.72
Wallet,3681749.35,391006.73,0.0,4072756.08



Sin los pagos de 50,000:


pay_method,card,cash,transfer,Total
product_type,,,,
Loan,486293.41,414826.31,0.0,901119.72
Card,0.0,0.0,815567.94,815567.94
Wallet,331749.35,391006.73,0.0,722756.08



Comparación de importes totales:


pay_method,total_con_50000,total_sin_50000,diferencia
product_type,,,
Card,6115567.94,815567.94,5300000.0
Loan,6101119.72,901119.72,5200000.0
Wallet,4072756.08,722756.08,3350000.0



3. TASA DE FALLOS POR SEGMENTO DE RIESGO


,total_pagos,pagos_fallidos,tasa_fallos_pct
risk_segment,,,
High,13344,2484,18.62
Medium,21850,4010,18.35
Low,58684,5745,9.79
Desconocido,6122,0,0.00


#### Comentario 
- Europa presenta la mayor tasa de pagos fallidos y contracargos, con un 14.80% (3,310 de 22,371 pagos), seguida del Caribe con un 14.71%. Norteamérica registra un 12.99% y la región Andina un 10.00%. La diferencia entre Europa y el Caribe es pequeña, de 0.09 puntos. Los 6,122 pagos sin región quedan pendiente de validación con el negocio.
- Incluir los importes de 50,000 cambia el producto con mayor importe total: Card lidera con todos los registros, mientras que Loan lidera al excluirlos. Esto demuestra que esos registros influyen en la conclusión.
- El segmento High presenta una tasa de fallos del 18.62%, ligeramente superior al 18.35% del segmento Medium. El segmento Low registra la tasa más baja, con un 9.79%. La diferencia entre High y Low es de 8.83 puntos. Los resultados muestran una asociación entre el segmento de riesgo y la tasa de fallos, aunque no demuestran causalidad. Los 6,122 registros con riesgo desconocido quedan pendiente de validación con el negocio.


### Revisión luego de corrección

In [365]:
importes = pd.to_numeric(
    df["payment_amount"],
    errors="coerce"
).dropna()

print("Estadísticas de importes:")
print(
    importes.describe(
        percentiles=[0.25, 0.50, 0.75, 0.99]
    )
)

print("\nImportes negativos:")
print(importes[importes < 0].value_counts().sort_index())

Estadísticas de importes:
count    93981.000000
mean       172.526428
std       2709.200510
min        -25.000000
25%         16.990000
50%         27.990000
75%         37.990000
99%         48.990000
max      50000.000000
Name: payment_amount, dtype: float64

Importes negativos:
payment_amount
-25.0    6328
Name: count, dtype: int64


In [366]:
print(df["payment_amount"].dtype)
montos = pd.to_numeric(df["payment_amount"], errors="coerce")
montos.describe().round(2)

str


count    93981.00
mean       172.53
std       2709.20
min        -25.00
25%         16.99
50%         27.99
75%         37.99
max      50000.00
Name: payment_amount, dtype: float64

In [367]:
montos = pd.to_numeric(df["payment_amount"], errors="coerce")

valores_invalidos = df.loc[
    montos.isna(),
    "payment_amount"
]

print("Total de valores no convertibles:", len(valores_invalidos))
print("\nEjemplos de valores:")
print(valores_invalidos.value_counts(dropna=False).head(20))

Total de valores no convertibles: 6019

Ejemplos de valores:
payment_amount
12,50    6019
Name: count, dtype: int64


In [368]:
montos = pd.to_numeric(
    df["payment_amount"].str.replace(",", ".", regex=False),
    errors="coerce"
)

print("Valores numéricos:", montos.notna().sum())
print("Valores no convertibles:", montos.isna().sum())

display(montos.describe().round(2))

Valores numéricos: 100000
Valores no convertibles: 0


count    100000.00
mean        162.89
std        2626.68
min         -25.00
25%          13.99
50%          26.99
75%          36.99
max       50000.00
Name: payment_amount, dtype: float64